In [1]:
# 0. IMPORTS
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    precision_score, recall_score, roc_auc_score, 
    average_precision_score, confusion_matrix, classification_report
)
import lightgbm as lgb

In [2]:
# 1. LOAD MODEL-READY DATASET FROM NOTEBOOK 03
df = pd.read_parquet('/kaggle/input/datasets/hamaz911/notebook-4-dataset/train_model_ready.parquet')
print(df.shape)

(590540, 315)


In [3]:
# 2. TIME-BASED SPLIT (NOT RANDOM) — sort by TransactionDT first
df = df.sort_values('TransactionDT').reset_index(drop=True)

split_point = int(len(df) * 0.8)
train = df.iloc[:split_point]
test = df.iloc[split_point:]

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("\nTrain date range (seconds):", train['TransactionDT'].min(), "-", train['TransactionDT'].max())
print("Test date range (seconds):", test['TransactionDT'].min(), "-", test['TransactionDT'].max())

print("\nTrain fraud rate:", train['isFraud'].mean() * 100)
print("Test fraud rate:", test['isFraud'].mean() * 100)

Train shape: (472432, 315)
Test shape: (118108, 315)

Train date range (seconds): 86400 - 12192842
Test date range (seconds): 12192900 - 15811131

Train fraud rate: 3.5135215226741625
Test fraud rate: 3.4409184813899145


In [4]:
# 3. SEPARATE FEATURES (X) AND TARGET (y)
drop_cols = ['isFraud', 'TransactionID', 'TransactionDT']  # ID/time columns aren't real features

X_train = train.drop(columns=drop_cols)
y_train = train['isFraud']

X_test = test.drop(columns=drop_cols)
y_test = test['isFraud']

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

X_train shape: (472432, 312)
X_test shape: (118108, 312)


In [5]:
# 4. QUICK PREP FOR LOGISTIC REGRESSION (needs numeric, non-null input)
# Convert categoricals to numeric codes, fill NaN with -999 (a flag value outside normal range)
X_train_lr = X_train.copy()
X_test_lr = X_test.copy()

cat_cols = X_train_lr.select_dtypes(include='category').columns
for col in cat_cols:
    X_train_lr[col] = X_train_lr[col].cat.codes
    X_test_lr[col] = X_test_lr[col].cat.codes

X_train_lr = X_train_lr.fillna(-999)
X_test_lr = X_test_lr.fillna(-999)

print("Any remaining NaN in train?", X_train_lr.isnull().sum().sum())

Any remaining NaN in train? 0


In [7]:
# 4b. TRAIN LOGISTIC REGRESSION BASELINE
lr_model = LogisticRegression(max_iter=1000, class_weight='balanced')
lr_model.fit(X_train_lr, y_train)

print("Model trained.")

ValueError: could not convert string to float: 'NotFound'

In [8]:
# 5a. DEBUG: FIND ANY REMAINING NON-NUMERIC COLUMNS
non_numeric = X_train_lr.select_dtypes(exclude=[np.number]).columns.tolist()
print("Non-numeric columns still remaining:", non_numeric)

# Check dtype of these columns
print(X_train_lr[non_numeric].dtypes)

Non-numeric columns still remaining: ['id_12', 'id_15', 'id_16', 'id_23', 'id_27', 'id_28', 'id_29', 'id_30', 'id_31', 'id_33', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38', 'DeviceInfo']
id_12         object
id_15         object
id_16         object
id_23         object
id_27         object
id_28         object
id_29         object
id_30         object
id_31         object
id_33         object
id_34         object
id_35         object
id_36         object
id_37         object
id_38         object
DeviceInfo    object
dtype: object


In [9]:
# 5b. FIX: DYNAMICALLY CONVERT ALL OBJECT + CATEGORY COLUMNS, NOT JUST OUR PREDEFINED LIST
X_train_lr = X_train.copy()
X_test_lr = X_test.copy()

# Catch both 'object' and 'category' dtypes this time
cat_cols_all = X_train_lr.select_dtypes(include=['object', 'category']).columns.tolist()
print(f"Found {len(cat_cols_all)} categorical/object columns to encode")

for col in cat_cols_all:
    X_train_lr[col] = X_train_lr[col].astype('category').cat.codes
    X_test_lr[col] = X_test_lr[col].astype('category').cat.codes

X_train_lr = X_train_lr.fillna(-999)
X_test_lr = X_test_lr.fillna(-999)

# VERIFY: confirm zero non-numeric columns remain
non_numeric_check = X_train_lr.select_dtypes(exclude=[np.number]).columns.tolist()
print("Any remaining non-numeric columns?", non_numeric_check)
print("Any remaining NaN?", X_train_lr.isnull().sum().sum())

Found 32 categorical/object columns to encode
Any remaining non-numeric columns? []
Any remaining NaN? 0


In [16]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_lr_scaled = scaler.fit_transform(X_train_lr)
X_test_lr_scaled = scaler.transform(X_test_lr)

lr_model = LogisticRegression(max_iter=5000, class_weight='balanced')
lr_model.fit(X_train_lr_scaled, y_train)

print("Model trained.")

Model trained.


In [17]:
# 6. EVALUATE LOGISTIC REGRESSION BASELINE
y_pred_lr = lr_model.predict(X_test_lr_scaled)
y_proba_lr = lr_model.predict_proba(X_test_lr_scaled)[:, 1]  # probability of fraud

print("Precision:", precision_score(y_test, y_pred_lr))
print("Recall:", recall_score(y_test, y_pred_lr))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_lr))
print("PR-AUC:", average_precision_score(y_test, y_proba_lr))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_lr))
print("\nFull Report:")
print(classification_report(y_test, y_pred_lr))

Precision: 0.11560203504804975
Recall: 0.7044783464566929
ROC-AUC: 0.8246613360757017
PR-AUC: 0.17292664894550364

Confusion Matrix:
[[92141 21903]
 [ 1201  2863]]

Full Report:
              precision    recall  f1-score   support

           0       0.99      0.81      0.89    114044
           1       0.12      0.70      0.20      4064

    accuracy                           0.80    118108
   macro avg       0.55      0.76      0.54    118108
weighted avg       0.96      0.80      0.86    118108



In [18]:
# 7. TRAIN RANDOM FOREST BASELINE
rf_model = RandomForestClassifier(
    n_estimators=100, 
    class_weight='balanced', 
    random_state=42,
    n_jobs=-1  # use all available CPU cores to speed up training
)
rf_model.fit(X_train_lr, y_train)

print("Model trained.")

Model trained.


In [19]:
# 8. EVALUATE RANDOM FOREST BASELINE
y_pred_rf = rf_model.predict(X_test_lr)
y_proba_rf = rf_model.predict_proba(X_test_lr)[:, 1]

print("Precision:", precision_score(y_test, y_pred_rf))
print("Recall:", recall_score(y_test, y_pred_rf))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_rf))
print("PR-AUC:", average_precision_score(y_test, y_proba_rf))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))
print("\nFull Report:")
print(classification_report(y_test, y_pred_rf))

Precision: 0.8717518860016764
Recall: 0.2559055118110236
ROC-AUC: 0.8907132777199269
PR-AUC: 0.5312779796218772

Confusion Matrix:
[[113891    153]
 [  3024   1040]]

Full Report:
              precision    recall  f1-score   support

           0       0.97      1.00      0.99    114044
           1       0.87      0.26      0.40      4064

    accuracy                           0.97    118108
   macro avg       0.92      0.63      0.69    118108
weighted avg       0.97      0.97      0.97    118108



In [20]:
# 9. TRAIN LIGHTGBM BASELINE (uses ORIGINAL X_train/X_test — native categorical + NaN handling)
lgb_model = lgb.LGBMClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42
)

lgb_model.fit(X_train, y_train)
print("Model trained.")

ValueError: pandas dtypes must be int, float or bool.
Fields with bad pandas dtypes: id_12: object, id_15: object, id_16: object, id_23: object, id_27: object, id_28: object, id_29: object, id_30: object, id_31: object, id_33: object, id_34: object, id_35: object, id_36: object, id_37: object, id_38: object, DeviceInfo: object

In [21]:
# 9a. FIX: CONVERT REMAINING OBJECT COLUMNS TO CATEGORY DTYPE (for LightGBM native handling)
remaining_object_cols = X_train.select_dtypes(include='object').columns.tolist()
print(f"Converting {len(remaining_object_cols)} object columns to category:", remaining_object_cols)

for col in remaining_object_cols:
    X_train[col] = X_train[col].astype('category')
    X_test[col] = X_test[col].astype('category')

# VERIFY: confirm no object dtypes remain
print("\nRemaining object columns:", X_train.select_dtypes(include='object').columns.tolist())

Converting 16 object columns to category: ['id_12', 'id_15', 'id_16', 'id_23', 'id_27', 'id_28', 'id_29', 'id_30', 'id_31', 'id_33', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38', 'DeviceInfo']

Remaining object columns: []


In [22]:
# 9b. TRAIN LIGHTGBM BASELINE 
lgb_model = lgb.LGBMClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42
)

lgb_model.fit(X_train, y_train)
print("Model trained.")

[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Number of positive: 16599, number of negative: 455833
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.609515 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 23804
[LightGBM] [Info] Number of data points in the train set: 472432, number of used features: 311
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
Model trained.


In [23]:
# QUICK CHECK: WHICH CATEGORICAL COLUMNS HAVE THE MOST UNIQUE VALUES?
cat_cols_check = X_train.select_dtypes(include='category').columns
for col in cat_cols_check:
    n_unique = X_train[col].nunique()
    if n_unique > 100:  # flag anything unusually high
        print(f"{col}: {n_unique} unique categories")

id_31: 110 unique categories
id_33: 212 unique categories
DeviceInfo: 1639 unique categories


In [24]:
# 9c. DROP ORIGINAL DeviceInfo (SUPERSEDED BY DeviceInfo_grouped FROM NOTEBOOK 03)
X_train = X_train.drop(columns=['DeviceInfo'])
X_test = X_test.drop(columns=['DeviceInfo'])

print("DeviceInfo_grouped still present?", 'DeviceInfo_grouped' in X_train.columns)
print("New X_train shape:", X_train.shape)

DeviceInfo_grouped still present? True
New X_train shape: (472432, 311)


In [25]:
# 9d. RETRAIN LIGHTGBM WITHOUT REDUNDANT DeviceInfo
lgb_model = lgb.LGBMClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42
)

lgb_model.fit(X_train, y_train)
print("Model trained.")

[LightGBM] [Info] Number of positive: 16599, number of negative: 455833
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.628093 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 23167
[LightGBM] [Info] Number of data points in the train set: 472432, number of used features: 310
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
Model trained.


In [26]:
# 10. EVALUATE LIGHTGBM BASELINE
y_pred_lgb = lgb_model.predict(X_test)
y_proba_lgb = lgb_model.predict_proba(X_test)[:, 1]

print("Precision:", precision_score(y_test, y_pred_lgb))
print("Recall:", recall_score(y_test, y_pred_lgb))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_lgb))
print("PR-AUC:", average_precision_score(y_test, y_proba_lgb))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_lgb))
print("\nFull Report:")
print(classification_report(y_test, y_pred_lgb))

Precision: 0.2095521974213226
Recall: 0.7438484251968503
ROC-AUC: 0.9043170330532048
PR-AUC: 0.5009432975219994

Confusion Matrix:
[[102641  11403]
 [  1041   3023]]

Full Report:
              precision    recall  f1-score   support

           0       0.99      0.90      0.94    114044
           1       0.21      0.74      0.33      4064

    accuracy                           0.89    118108
   macro avg       0.60      0.82      0.63    118108
weighted avg       0.96      0.89      0.92    118108



In [27]:
# SAVE 1: THE METRICS COMPARISON TABLE (for our report/README later)
import json

results = {
    'logistic_regression': {'precision': 0.1156, 'recall': 0.7045, 'roc_auc': 0.8247, 'pr_auc': 0.1729},
    'random_forest': {'precision': 0.8718, 'recall': 0.2559, 'roc_auc': 0.8907, 'pr_auc': 0.5313},
    'lightgbm': {'precision': 0.2096, 'recall': 0.7438, 'roc_auc': 0.9043, 'pr_auc': 0.5009}
}

with open('/kaggle/working/baseline_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print("Saved baseline results.")

Saved baseline results.


In [28]:
# SAVE 2: THE LIGHTGBM BASELINE MODEL ITSELF (our reference point before tuning)
import joblib

joblib.dump(lgb_model, '/kaggle/working/lgb_baseline_model.pkl')
print("Saved LightGBM baseline model.")

Saved LightGBM baseline model.


In [29]:
# SAVE 3: THE EXACT TRAIN/TEST SPLIT (so Notebook 05/06 use IDENTICAL data — critical for fair comparison)
X_train.to_parquet('/kaggle/working/X_train.parquet', index=False)
X_test.to_parquet('/kaggle/working/X_test.parquet', index=False)
y_train.to_frame().to_parquet('/kaggle/working/y_train.parquet', index=False)
y_test.to_frame().to_parquet('/kaggle/working/y_test.parquet', index=False)

print("Saved train/test split.")

Saved train/test split.
